# Прогнозирование задержек доставки Olist

Датасет содержит несколько связанных таблиц с информацией о заказах бразильской платформы электронной коммерции Olist.

Цель этого ноутбука — изучить структуру данных, понять назначение и связи таблиц, а также определить, какие данные можно использовать для прогнозирования задержек доставки.

Начнём с таблицы `orders`, в которой одна строка соответствует одному заказу.


### Импорт библиотек и загрузка данных

In [20]:
import pandas as pd
import numpy as np

In [3]:
orders = pd.read_csv('../data/raw/olist_orders_dataset.csv', parse_dates=['order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date',
                                                                         'order_delivered_customer_date', 'order_estimated_delivery_date'])

In [4]:
#выводим общую информацию по таблице
print('shape:', orders.shape)
print('column names:', orders.columns)
print('dtypes:', orders.dtypes)

shape: (99441, 8)
column names: Index(['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp',
       'order_approved_at', 'order_delivered_carrier_date',
       'order_delivered_customer_date', 'order_estimated_delivery_date'],
      dtype='object')
dtypes: order_id                                 object
customer_id                              object
order_status                             object
order_purchase_timestamp         datetime64[ns]
order_approved_at                datetime64[ns]
order_delivered_carrier_date     datetime64[ns]
order_delivered_customer_date    datetime64[ns]
order_estimated_delivery_date    datetime64[ns]
dtype: object


Таблица состоит из 99441 строк и 8 столбцов. Из 8, 5 колонок содержат даты.

In [5]:
orders.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26


In [6]:
print(orders.duplicated('order_id').sum())

0


`order_id` не имеет дубликатов и является уникальным для каждой строки

In [7]:
print(orders.order_status.value_counts())

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64


Большинство заказов имеют статус delivered — 96 478 из 99 441. Также присутствуют отменённые, недоступные и незавершённые заказы, поэтому перед созданием целевой переменной потребуется определить, какие статусы включать в выборку.

In [8]:
missing_values = pd.DataFrame({'Количество пропусков' : orders.isna().sum(), 
                               'Доля пропусков' : (orders.isna().mean() * 100).round(2)})
missing_values.index.name = 'Столбец'
missing_values

,Количество пропусков,Доля пропусков
Столбец,,
order_id,0,0.00
customer_id,0,0.00
order_status,0,0.00
order_purchase_timestamp,0,0.00
order_approved_at,160,0.16
order_delivered_carrier_date,1783,1.79
order_delivered_customer_date,2965,2.98
order_estimated_delivery_date,0,0.00


Пропуски встречаются только во временных столбцах. Больше всего их в фактической дате доставки покупателю - 2965 значений. Если вычесть количество заказов статус которых указан как доставленный из общего количества, то получится очень близкое к этому значение - 2963. Необходимо проверить, какими статусами объясняются эти пропуски.

In [9]:
orders.groupby('order_status')['order_delivered_customer_date'].agg(всего_заказов='size',
                                                                   пропусков=lambda x: x.isna().sum(),
                                                                   доля_пропусков_проц=lambda x: (x.isna().mean() * 100).round(2))

,всего_заказов,пропусков,доля_пропусков_проц
order_status,,,
approved,2,2,100.00
canceled,625,619,99.04
created,5,5,100.00
delivered,96478,8,0.01
invoiced,314,314,100.00
processing,301,301,100.00
shipped,1107,1107,100.00
unavailable,609,609,100.00


Почти все пропуски относятся к заказам, не обозначенным как доставленные, что логично, ведь у недоставленного заказа не может быть даты доставки, кроме предполагаемой. При этом 8 доставленных заказов все равно имеют пропуски в дате

In [47]:
dates = orders.drop(columns=['order_id', 'customer_id', 'order_status'])
print(dates.agg(['min', 'max']).T)

                                              min                 max
Столбец                                                              
order_purchase_timestamp      2016-09-04 21:15:19 2018-10-17 17:30:18
order_approved_at             2016-09-15 12:16:38 2018-09-03 17:40:06
order_delivered_carrier_date  2016-10-08 10:34:01 2018-09-11 19:48:28
order_delivered_customer_date 2016-10-11 13:46:32 2018-10-17 13:22:46
order_estimated_delivery_date 2016-09-30 00:00:00 2018-11-12 00:00:00


Проверка минимальных и максимальных дат не выявила необычных значений. Данные охватывают примерно один период — с сентября 2016 года по ноябрь 2018 года. Максимальная ожидаемая дата доставки позже максимальной фактической даты, что логично: для последних заказов дата доставки могла быть запланирована на более поздний срок.
Максимальные даты промежуточных этапов, таких как подтверждение заказа и передача перевозчику, могут быть раньше максимальной даты покупки: последние заказы в датасете могли ещё не пройти эти этапы или соответствующие события не были зафиксированы.

In [48]:
for i in range(dates.shape[1] - 1):
    date_pair = dates.iloc[:, i:i+2].dropna()
    print((date_pair.iloc[:, 0] > date_pair.iloc[:, 1]).sum())


0
1359
23
7827


В данных обнаружены отдельные нарушения последовательности логистических дат: в 1 359 заказах передача перевозчику указана раньше подтверждения, ещё в 23 случаях доставка покупателю произошла раньше передачи перевозчику. Эти записи потребуют отдельного решения при подготовке данных. Кроме того, в 7 827 заказах фактическая дата доставки позже обещанной — такие заказы будут относиться к классу задержанных.

Таблица orders содержит 99 441 заказ и используется как основа для формирования целевой переменной. Идентификатор order_id уникален, поэтому одна строка соответствует одному заказу.

Проверка дат не выявила нарушений порядка фактических этапов обработки заказа. Для почти всех заказов со статусом delivered доступны фактическая и ожидаемая даты доставки: исключением являются только 8 заказов без фактической даты. Следовательно, для остальных доставленных заказов можно определить, была ли доставка выполнена с задержкой.

Следующая таблица - `customers`. Как и таблица `orders`, она содержит 99441 строку, так как она связана с ней через уникальный для каждого заказа `customer_id`. Также для каждого пользователя есть `customer_unique_id`. Они уже могут повторяться, так как покупатель может сделать несколько заказов.
Помимо идентификаторов, в таблице содержаться географические данные заказчиков: почтовый код, город и регион.

In [12]:
customers = pd.read_csv('../data/raw/olist_customers_dataset.csv')

In [13]:
print('shape:', customers.shape)
print('column names:', customers.columns)
print('dtypes:', customers.dtypes)

shape: (99441, 5)
column names: Index(['customer_id', 'customer_unique_id', 'customer_zip_code_prefix',
       'customer_city', 'customer_state'],
      dtype='object')
dtypes: customer_id                 object
customer_unique_id          object
customer_zip_code_prefix     int64
customer_city               object
customer_state              object
dtype: object


In [14]:
customers.head()

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP


In [15]:
print(customers.duplicated('customer_id').sum())
print(customers.duplicated('customer_unique_id').sum())

0
3345


In [16]:
print(customers["customer_city"].nunique())
print(customers['customer_zip_code_prefix'].nunique())
print(customers['customer_state'].nunique())

4119
14994
27


In [17]:
orders["customer_id"].isin(customers["customer_id"]).all()
customers.isna().sum()

customer_id                 0
customer_unique_id          0
customer_zip_code_prefix    0
customer_city               0
customer_state              0
dtype: int64

В таблице `customers` отсутствуют пропуски. `customer_id` уникален и связывает запись покупателя с конкретным заказом, тогда как `customer_unique_id` может повторяться для нескольких заказов одного покупателя.
Географические признаки имеют разную кардинальность: штат содержит небольшое число категорий, тогда как город и ZIP-префикс — тысячи уникальных значений. Поэтому в первой версии модели будет использован штат покупателя; город и ZIP-префикс пока не включаются без дополнительной обработки.

Таблица order_items содержит информацию о товарных позициях в заказах: товаре, продавце, цене, стоимости доставки и сроке передачи заказа продавцом. В отличие от orders, один order_id здесь может встречаться несколько раз, поскольку один заказ может включать несколько товаров.

In [21]:
order_items = pd.read_csv('../data/raw/olist_order_items_dataset.csv')

In [24]:
print('shape:', order_items.shape)
print('column names:', order_items.columns)
print('dtypes:', order_items.dtypes)

shape: (112650, 7)
column names: Index(['order_id', 'order_item_id', 'product_id', 'seller_id',
       'shipping_limit_date', 'price', 'freight_value'],
      dtype='object')
dtypes: order_id                object
order_item_id            int64
product_id              object
seller_id               object
shipping_limit_date     object
price                  float64
freight_value          float64
dtype: object


In [26]:
order_items.head()

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14


In [27]:
order_items.isna().sum()

order_id               0
order_item_id          0
product_id             0
seller_id              0
shipping_limit_date    0
price                  0
freight_value          0
dtype: int64

In [29]:
print(order_items['order_id'].duplicated().sum())

13984


In [31]:
items_per_order = order_items.groupby("order_id").size()
print((items_per_order > 1).sum())

9803


In [45]:
items_per_order = order_items.groupby("order_id")['seller_id'].nunique()
print((items_per_order > 1).sum())

1278


In [46]:
order_items["order_id"].isin(orders["order_id"]).all()

np.True_

В таблице order_items отсутствуют пропуски. Она содержит 112 650 товарных позиций для заказов из таблицы orders. Повторение order_id ожидаемо: 9 803 заказа содержат более одной позиции. Кроме того, 1 278 заказов включают товары от нескольких продавцов.

При объединении с таблицей orders данные потребуется агрегировать до уровня заказа: например, посчитать число товаров, суммарную стоимость, суммарную стоимость доставки и число продавцов.

Таблица products содержит характеристики товаров: категорию, название и описание, а также вес и габариты. Она связывается с order_items через product_id и позволит добавить к заказу признаки, связанные с особенностями перевозки товаров.

In [49]:
products = pd.read_csv('../data/raw/olist_products_dataset.csv')

In [50]:
print('shape:', products.shape)
print('column names:', products.columns)
print('dtypes:', products.dtypes)

shape: (32951, 9)
column names: Index(['product_id', 'product_category_name', 'product_name_lenght',
       'product_description_lenght', 'product_photos_qty', 'product_weight_g',
       'product_length_cm', 'product_height_cm', 'product_width_cm'],
      dtype='object')
dtypes: product_id                     object
product_category_name          object
product_name_lenght           float64
product_description_lenght    float64
product_photos_qty            float64
product_weight_g              float64
product_length_cm             float64
product_height_cm             float64
product_width_cm              float64
dtype: object


In [54]:
print(products['product_id'].nunique())

32951


In [55]:
products.head()

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0
3,cef67bcfe19066a932b7673e239eb23d,bebes,27.0,261.0,1.0,371.0,26.0,4.0,26.0
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37.0,402.0,4.0,625.0,20.0,17.0,13.0


In [56]:
products.isna().sum()

product_id                      0
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64

In [57]:
print(order_items['product_id'].isin(products['product_id']).all())

True


In [59]:
products['product_category_name'].nunique()

73

In [64]:
products[['product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']].describe()

,product_weight_g,product_length_cm,product_height_cm,product_width_cm
count,32949.000000,32949.000000,32949.000000,32949.000000
mean,2276.472488,30.815078,16.937661,23.196728
std,4282.038731,16.914458,13.637554,12.079047
min,0.000000,7.000000,2.000000,6.000000
25%,300.000000,18.000000,8.000000,15.000000
50%,700.000000,25.000000,13.000000,20.000000
75%,1900.000000,38.000000,21.000000,30.000000
max,40425.000000,105.000000,105.000000,118.000000


Таблица `products` содержит 32 951 уникальный товар; для всех товаров из `order_items` найдены соответствующие записи.

В 610 строках отсутствуют одновременно категория, длина названия и описания, количество фотографий. Ещё в 2 строках отсутствуют физические характеристики товара. Всего представлено 73 товарные категории.

Вес товаров варьируется от 0 до 40 425 г, размеры — от 7×2×6 до 105×105×118 см. Нулевое значение встречается только у веса.

Для прогнозирования потенциально полезны категория товара, вес и габариты, поскольку они могут влиять на особенности перевозки и срок доставки.

In [65]:
sellers = pd.read_csv('../data/raw/olist_sellers_dataset.csv')

In [66]:
print('shape:', sellers.shape)
print('column names:', sellers.columns)
print('dtypes:', sellers.dtypes)

shape: (3095, 4)
column names: Index(['seller_id', 'seller_zip_code_prefix', 'seller_city', 'seller_state'], dtype='object')
dtypes: seller_id                 object
seller_zip_code_prefix     int64
seller_city               object
seller_state              object
dtype: object


In [67]:
sellers.head()

,seller_id,seller_zip_code_prefix,seller_city,seller_state
0,3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP
2,ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ
3,c0f3eea2e14555b6faeea3dd58c1b1c3,4195,sao paulo,SP
4,51a04a8a6bdcb23deccc82b0b80742cf,12914,braganca paulista,SP


In [68]:
sellers.isna().sum()

seller_id                 0
seller_zip_code_prefix    0
seller_city               0
seller_state              0
dtype: int64

In [69]:
sellers['seller_id'].nunique()

3095

In [71]:
print(order_items['seller_id'].isin(sellers['seller_id']).all())

True


Таблица sellers содержит информацию о географии 3 095 продавцов. В ней отсутствуют пропуски, а seller_id уникален. Для всех продавцов из order_items найдены соответствующие записи в таблице.

В первой версии модели можно использовать штат продавца как географический признак. Город и ZIP-префикс, как и у покупателей, пока не включаем без дополнительной обработки из-за большого числа возможных значений.

В ходе первичного анализа были рассмотрены основные таблицы, необходимые для прогнозирования задержек доставки: orders, customers, order_items, products и sellers.

Целевая переменная будет определяться по сравнению фактической и обещанной дат доставки. Для первой версии модели можно использовать данные о покупателе, составе заказа, товарах и продавцах. При этом признаки, появляющиеся уже после оформления заказа — например фактические даты передачи и доставки — использовать нельзя, так как это приведёт к утечке данных.

В данных обнаружены пропуски и отдельные нарушения последовательности логистических дат; способ их обработки будет определён на следующем этапе подготовки данных.